## 1 — Imports and Configuration

In [1]:
import os
import requests
import chess
import chess.pgn
import io
import random
import boto3
from pathlib import Path

# Configuration
BUCKET_NAME = "chess-pgn-capstone-387100404445-us-east-1-an"
NUM_GAMES    = 100000  # Total clean games to download
MIN_MOVES    = 20      # Ignore very short games
MAX_MOVES    = 100     # Ignore very long games (keeps positions manageable)
RANDOM_SEED  = 42

random.seed(RANDOM_SEED)

print("Configuration loaded successfully")
print(f"Target games : {NUM_GAMES:,}")
print(f"Move filter  : {MIN_MOVES} to {MAX_MOVES} moves")
print(f"S3 bucket    : {BUCKET_NAME}")

Configuration loaded successfully
Target games : 100,000
Move filter  : 20 to 100 moves
S3 bucket    : chess-pgn-capstone-387100404445-us-east-1-an


## 2 — Download Lichess PGN File

In [2]:
# Install zstandard decompressor
import subprocess
subprocess.run(["pip", "install", "zstandard", "-q"])
import zstandard as zstd
print("zstandard installed OK")

zstandard installed OK


## 3 - Download and Decompress Lichess file

In [3]:
# Download the compressed Lichess file
LICHESS_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2013-01.pgn.zst"
LOCAL_ZST   = "/home/sagemaker-user/lichess_2013_01.pgn.zst"
LOCAL_PGN   = "/home/sagemaker-user/lichess_2013_01.pgn"

print("Downloading Lichess database file...")
print("This may take 2-5 minutes...")

response = requests.get(LICHESS_URL, stream=True)
response.raise_for_status()

total_bytes = 0
with open(LOCAL_ZST, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        total_bytes += len(chunk)

print(f"Download complete — {total_bytes / (1024*1024):.1f} MB")

# Decompress the .zst file
print("\nDecompressing...")
dctx = zstd.ZstdDecompressor()
with open(LOCAL_ZST, 'rb') as compressed:
    with open(LOCAL_PGN, 'wb') as destination:
        dctx.copy_stream(compressed, destination)

pgn_size = os.path.getsize(LOCAL_PGN) / (1024*1024)
print(f"Decompressed PGN size: {pgn_size:.1f} MB")
print(f"Saved to: {LOCAL_PGN}")

This may take 2-5 minutes...


Download complete — 16.9 MB

Decompressing...


Decompressed PGN size: 88.5 MB
Saved to: /home/sagemaker-user/lichess_2013_01.pgn


## 4 — Split Into Individual PGN Files

In [4]:
# Split the large PGN file into individual game files
OUTPUT_DIR = "/home/sagemaker-user/raw_pgn_games/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Reading and splitting PGN file into individual game files...")
print(f"Filters: {MIN_MOVES} to {MAX_MOVES} moves per game")
print(f"Target : {NUM_GAMES:,} games")
print("This may take 3-5 minutes...")

games_saved  = 0
games_read   = 0
games_skipped = 0

with open(LOCAL_PGN, encoding="utf-8", errors="ignore") as pgn_file:
    while games_saved < NUM_GAMES:
        game = chess.pgn.read_game(pgn_file)
        if game is None:
            print(f"Reached end of file after reading {games_read:,} games")
            break

        games_read += 1

        # Count moves in this game
        num_moves = game.end().ply()

        # Skip games outside our move range
        if num_moves < MIN_MOVES or num_moves > MAX_MOVES:
            games_skipped += 1
            continue

        # Save as individual PGN file
        filename = f"{OUTPUT_DIR}game_{games_saved + 1:06d}.pgn"
        with open(filename, "w") as out_file:
            print(game, file=out_file, end="\n\n")

        games_saved += 1

        # Progress update every 10,000 games
        if games_saved % 10000 == 0:
            print(f"  Saved {games_saved:,} games so far...")

print(f"\nDone!")
print(f"Total games read    : {games_read:,}")
print(f"Games skipped       : {games_skipped:,} (outside move range)")
print(f"Individual PGN files: {games_saved:,}")
print(f"Saved to            : {OUTPUT_DIR}")

Reading and splitting PGN file into individual game files...
Filters: 20 to 100 moves per game
Target : 100,000 games
This may take 3-5 minutes...


  Saved 10,000 games so far...


  Saved 20,000 games so far...


  Saved 30,000 games so far...


  Saved 40,000 games so far...


...

  Saved 60,000 games so far...


  Saved 70,000 games so far...


  Saved 80,000 games so far...


  Saved 90,000 games so far...


Reached end of file after reading 121,332 games

Done!
Total games read    : 121,332
Games skipped       : 24,172 (outside move range)
Individual PGN files: 97,160
Saved to            : /home/sagemaker-user/raw_pgn_games/


## 5 - Quick Sanity Check

In [5]:
# Read one sample game to verify individual files are correct
sample_file = f"{OUTPUT_DIR}game_000001.pgn"

with open(sample_file) as f:
    content = f.read()
    
print("Sample PGN file content:")
print("=" * 50)
print(content)
print("=" * 50)

# Also verify move count
with open(sample_file) as f:
    game = chess.pgn.read_game(f)
    
num_moves = game.end().ply()
print(f"Number of half-moves (plies): {num_moves}")
print(f"Number of full moves        : {num_moves // 2}")
print(f"White player : {game.headers.get('White', 'Unknown')}")
print(f"Black player : {game.headers.get('Black', 'Unknown')}")
print(f"Result       : {game.headers.get('Result', 'Unknown')}")

Sample PGN file content:
[Event "Rated Classical game"]
[Site "https://lichess.org/j1dkb5dw"]
[Date "????.??.??"]
[Round "?"]
[White "BFG9k"]
[Black "mamalak"]
[Result "1-0"]
[UTCDate "2012.12.31"]
[UTCTime "23:01:03"]
[WhiteElo "1639"]
[BlackElo "1403"]
[WhiteRatingDiff "+5"]
[BlackRatingDiff "-8"]
[ECO "C00"]
[Opening "French Defense: Normal Variation"]
[TimeControl "600+8"]
[Termination "Normal"]

1. e4 e6 2. d4 b6 3. a3 Bb7 4. Nc3 Nh6 5. Bxh6 gxh6 6. Be2 Qg5 7. Bg4 h5 8. Nf3 Qg6 9. Nh4 Qg5 10. Bxh5 Qxh4 11. Qf3 Kd8 12. Qxf7 Nc6 13. Qe8# 1-0


Number of half-moves (plies): 25
Number of full moves        : 12
White player : BFG9k
Black player : mamalak
Result       : 1-0


## 6 - Update the Dataset Statistics

In [6]:
import glob

# Count all individual PGN files
all_files = glob.glob(f"{OUTPUT_DIR}*.pgn")
total_files = len(all_files)

# Sample 5 random games and show their move counts
print(f"Total individual PGN files: {total_files:,}")
print(f"\nSample of 5 random games:")
print("-" * 40)

sample_files = random.sample(all_files, 5)
for filepath in sample_files:
    with open(filepath) as f:
        game = chess.pgn.read_game(f)
    plies     = game.end().ply()
    full_moves = plies // 2
    result    = game.headers.get('Result', '?')
    opening   = game.headers.get('Opening', 'Unknown')
    print(f"File     : {os.path.basename(filepath)}")
    print(f"Moves    : {full_moves} full moves ({plies} plies)")
    print(f"Result   : {result}")
    print(f"Opening  : {opening}")
    print("-" * 40)

Total individual PGN files: 97,160

Sample of 5 random games:
----------------------------------------
File     : game_083811.pgn
Moves    : 45 full moves (91 plies)
Result   : 1-0
Opening  : King's Indian Defense: Normal Variation, King's Knight Variation
----------------------------------------
File     : game_014593.pgn
Moves    : 31 full moves (63 plies)
Result   : 1-0
Opening  : Bishop's Opening: Boi Variation
----------------------------------------
File     : game_003279.pgn
Moves    : 31 full moves (63 plies)
Result   : 1-0
Opening  : King's Pawn Opening: 2.b3
----------------------------------------
File     : game_036049.pgn
Moves    : 42 full moves (84 plies)
Result   : 0-1
Opening  : Englund Gambit Complex: Hartlaub-Charlick Gambit
----------------------------------------
File     : game_032099.pgn
Moves    : 19 full moves (38 plies)
Result   : 0-1
Opening  : Bird Opening: Dutch Variation
----------------------------------------


## 7 - Upload Raw PGN Files to S3

In [20]:
import os
import glob
import boto3

s3 = boto3.client('s3', region_name='us-east-1')
BUCKET = "chess-pgn-capstone-387100404445-us-east-1-an"

RAW_PGN_LOCAL = "/home/sagemaker-user/raw_pgn_games/"

print("Uploading raw PGN files to S3...")
print("This may take 15-20 minutes...")
print()

files = sorted(glob.glob(f"{RAW_PGN_LOCAL}*.pgn"))
total = len(files)
uploaded = 0
errors = 0

for filepath in files:
    try:
        filename = os.path.basename(filepath)
        s3_key = f"raw-pgn/{filename}"
        s3.upload_file(filepath, BUCKET, s3_key)
        uploaded += 1
        
        if uploaded % 10000 == 0:
            print(f"  Uploaded {uploaded:,} of {total:,}...")
    
    except Exception as e:
        errors += 1
        continue

print()
print("=" * 50)
print("RAW PGN UPLOAD COMPLETE")
print("=" * 50)
print(f"Total files uploaded : {uploaded:,}")
print(f"Errors               : {errors}")
print(f"S3 location          : s3://{BUCKET}/raw-pgn/")

Uploading raw PGN files to S3...
This may take 15-20 minutes...



  Uploaded 10,000 of 97,160...


  Uploaded 20,000 of 97,160...


  Uploaded 30,000 of 97,160...


  Uploaded 40,000 of 97,160...


...

  Uploaded 60,000 of 97,160...


  Uploaded 70,000 of 97,160...


  Uploaded 80,000 of 97,160...


  Uploaded 90,000 of 97,160...



RAW PGN UPLOAD COMPLETE
Total files uploaded : 97,160
Errors               : 0
S3 location          : s3://chess-pgn-capstone-387100404445-us-east-1-an/raw-pgn/


## 8 - Build the Corruption Pipeline

In [13]:
import random
import copy

def corrupt_pgn(game, error_type=None):
    """
    Apply one of four error types to a chess game.
    
    Error Types:
        1 - Wrong move notation (illegal move substitution)
        2 - OCR character substitution
        3 - Move deletion without column shift
        4 - Move deletion with column shift cascade
    
    Returns:
        corrupted_pgn_string : the corrupted game as PGN text
        error_info           : dict describing what was done
    """
    
    # Collect all SAN moves from the game
    board = game.board()
    san_moves = []
    for move in game.mainline_moves():
        san_moves.append(board.san(move))
        board.push(move)
    
    total_moves = len(san_moves)
    
    if total_moves < 10:
        return None, None  # Skip very short games
    
    # Pick random error type if not specified
    if error_type is None:
        error_type = random.randint(1, 4)
    
    # Pick a random injection point — avoid first 3 and last 3 moves
    inject_at = random.randint(3, total_moves - 4)
    
    error_info = {
        "error_type"      : error_type,
        "inject_at_move"  : inject_at,
        "total_moves"     : total_moves,
        "original_move"   : san_moves[inject_at]
    }
    
    # Work on a copy of moves
    corrupted_moves = san_moves.copy()
    
    # ── TYPE 1: Wrong move notation (illegal move) ──────────────────────
    if error_type == 1:
        illegal_moves = ["Qh8", "Rb9", "Nz5", "Bh9", "Ka9", "Qа9"]
        bad_move = random.choice(illegal_moves)
        corrupted_moves[inject_at] = bad_move
        error_info["injected_move"] = bad_move
    
    # ── TYPE 2: OCR character substitution ──────────────────────────────
    elif error_type == 2:
        original_move = san_moves[inject_at]
        corrupted_move = original_move
        
        # OCR confusion map
        ocr_map = {
            'B': '8', 'O': '0', 'l': '1',
            'Q': 'O', 'N': 'M', 'R': 'P',
            'K': 'X', '0': 'O', '1': 'l'
        }
        
        # Try to substitute a character using OCR map
        substituted = False
        for i, char in enumerate(original_move):
            if char in ocr_map:
                corrupted_move = (original_move[:i] + 
                                  ocr_map[char] + 
                                  original_move[i+1:])
                substituted = True
                break
        
        # If no OCR substitution found flip last character
        if not substituted:
            last = original_move[-1]
            if last.isdigit():
                new_digit = str((int(last) % 8) + 1)
                corrupted_move = original_move[:-1] + new_digit
            else:
                corrupted_move = original_move[:-1] + "9"
        
        corrupted_moves[inject_at] = corrupted_move
        error_info["corrupted_move"] = corrupted_move
    
    # ── TYPE 3: Move deletion without column shift ───────────────────────
    elif error_type == 3:
        deleted_move = san_moves[inject_at]
        corrupted_moves.pop(inject_at)
        error_info["deleted_move"]   = deleted_move
        error_info["deletion_point"] = inject_at
    
    # ── TYPE 4: Move deletion with column shift cascade ──────────────────
    elif error_type == 4:
        deleted_move = san_moves[inject_at]
        corrupted_moves.pop(inject_at)
        error_info["deleted_move"]    = deleted_move
        error_info["shift_from_move"] = inject_at
        error_info["cascade_length"]  = total_moves - inject_at - 1
    
    # ── Build corrupted PGN string ───────────────────────────────────────
    pgn_lines = []
    
    # Copy original headers
    for key, value in game.headers.items():
        pgn_lines.append(f'[{key} "{value}"]')
    
    # Add corruption metadata headers
    pgn_lines.append(f'[CorruptionType "{error_type}"]')
    pgn_lines.append(f'[CorruptionAt "{inject_at}"]')
    pgn_lines.append("")
    
    # Build move text with correct move numbers
    move_text = ""
    for i, san in enumerate(corrupted_moves):
        full_move_num = (i // 2) + 1
        if i % 2 == 0:
            move_text += f"{full_move_num}. {san} "
        else:
            move_text += f"{san} "
    
    move_text += game.headers.get("Result", "*")
    pgn_lines.append(move_text.strip())
    
    return "\n".join(pgn_lines), error_info


print("Fixed corruption pipeline loaded successfully")

Fixed corruption pipeline loaded successfully


## 9 - Test All Four Error Types

In [15]:
# Test each error type on a sample game
test_file = f"{OUTPUT_DIR}game_000010.pgn"

print("=" * 60)
print("CORRUPTION PIPELINE TEST")
print("=" * 60)

for error_type in [1, 2, 3, 4]:
    print(f"\n── Error Type {error_type} ──────────────────────────────────")
    
    # Read the clean game
    with open(test_file) as f:
        game = chess.pgn.read_game(f)
    
    # Show original first 10 moves
    board = game.board()
    original_moves = []
    node = game
    while node.variations:
        next_node = node.variations[0]
        original_moves.append(board.san(next_node.move))
        board.push(next_node.move)
        node = next_node
    
    print(f"Original moves (first 10): {' '.join(original_moves[:10])}")
    
    # Apply corruption
    with open(test_file) as f:
        game = chess.pgn.read_game(f)
    
    corrupted_pgn, error_info = corrupt_pgn(game, error_type=error_type)
    
    if corrupted_pgn is None:
        print("Game too short — skipped")
        continue
    
    # Extract corrupted moves from PGN string
    pgn_move_line = [l for l in corrupted_pgn.split('\n') 
                     if l and not l.startswith('[')]
    
    print(f"Error info      : {error_info}")
    print(f"Corrupted moves : {' '.join(pgn_move_line)[:120]}...")

print("\n" + "=" * 60)
print("All four error types tested successfully")
print("=" * 60)

CORRUPTION PIPELINE TEST

── Error Type 1 ──────────────────────────────────
Original moves (first 10): d4 e5 dxe5 d6 exd6 Bxd6 Nf3 Nf6 Nc3 O-O
Error info      : {'error_type': 1, 'inject_at_move': 16, 'total_moves': 90, 'original_move': 'O-O', 'injected_move': 'Qа9'}
Corrupted moves : 1. d4 e5 2. dxe5 d6 3. exd6 Bxd6 4. Nf3 Nf6 5. Nc3 O-O 6. a3 Nc6 7. e3 a6 8. Be2 h6 9. Qа9 Ne5 10. Bd2 Nxf3+ 11. Bxf3 Be...

── Error Type 2 ──────────────────────────────────
Original moves (first 10): d4 e5 dxe5 d6 exd6 Bxd6 Nf3 Nf6 Nc3 O-O
Error info      : {'error_type': 2, 'inject_at_move': 72, 'total_moves': 90, 'original_move': 'Qe8+', 'corrupted_move': 'Oe8+'}
Corrupted moves : 1. d4 e5 2. dxe5 d6 3. exd6 Bxd6 4. Nf3 Nf6 5. Nc3 O-O 6. a3 Nc6 7. e3 a6 8. Be2 h6 9. O-O Ne5 10. Bd2 Nxf3+ 11. Bxf3 Be...

── Error Type 3 ──────────────────────────────────
Original moves (first 10): d4 e5 dxe5 d6 exd6 Bxd6 Nf3 Nf6 Nc3 O-O
Error info      : {'error_type': 3, 'inject_at_move': 14, 'total_moves': 90, 'ori

## 10 - Run Full Corruption Pipeline

In [16]:
import os
import glob
import json
import csv
import chess
import chess.pgn
import random
import shutil

# ── Configuration ────────────────────────────────────────────
ALL_GAMES_DIR    = "/home/sagemaker-user/raw_pgn_games/"
CORRUPTED_DIR    = "/home/sagemaker-user/corrupted_pgn_games/"
GROUND_TRUTH_DIR = "/home/sagemaker-user/ground_truth_games/"
CLEAN_DIR        = "/home/sagemaker-user/clean_pgn_games/"
METADATA_FILE    = "/home/sagemaker-user/corruption_metadata.csv"

CORRUPT_RATIO    = 0.80
RANDOM_SEED      = 42

random.seed(RANDOM_SEED)

# ── Create output directories ────────────────────────────────
os.makedirs(CORRUPTED_DIR,    exist_ok=True)
os.makedirs(GROUND_TRUTH_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR,        exist_ok=True)

# ── Get all game files and shuffle ──────────────────────────
all_files = sorted(glob.glob(f"{ALL_GAMES_DIR}*.pgn"))
random.shuffle(all_files)

total_games    = len(all_files)
n_corrupt      = int(total_games * CORRUPT_RATIO)
n_clean        = total_games - n_corrupt

corrupt_files  = all_files[:n_corrupt]
clean_files    = all_files[n_corrupt:]

print(f"Total games       : {total_games:,}")
print(f"To corrupt (80%)  : {n_corrupt:,}")
print(f"To keep clean (20%): {n_clean:,}")
print(f"\nError type distribution within corrupted set:")
print(f"  Type 1 — Wrong move    : ~{n_corrupt // 4:,}")
print(f"  Type 2 — OCR sub       : ~{n_corrupt // 4:,}")
print(f"  Type 3 — Delete no shift: ~{n_corrupt // 4:,}")
print(f"  Type 4 — Delete + shift : ~{n_corrupt // 4:,}")
print(f"\nStarting pipeline...")

# ── Assign error types evenly across corrupted files ────────
error_types = []
for i in range(n_corrupt):
    error_types.append((i % 4) + 1)
random.shuffle(error_types)

# ── Process corrupted files ──────────────────────────────────
metadata_rows  = []
success_count  = 0
skip_count     = 0
error_count    = 0

for idx, (filepath, error_type) in enumerate(zip(corrupt_files, error_types)):
    try:
        # Read clean game
        with open(filepath, encoding="utf-8", errors="ignore") as f:
            game = chess.pgn.read_game(f)
        
        if game is None:
            skip_count += 1
            continue
        
        # Apply corruption
        corrupted_pgn, error_info = corrupt_pgn(game, error_type=error_type)
        
        if corrupted_pgn is None:
            skip_count += 1
            continue
        
        # Generate output filenames
        base_name = f"game_{success_count + 1:06d}.pgn"
        
        # Save corrupted file
        with open(f"{CORRUPTED_DIR}{base_name}", "w") as f:
            f.write(corrupted_pgn)
        
        # Save ground truth (original clean game)
        with open(f"{GROUND_TRUTH_DIR}{base_name}", "w") as f:
            print(game, file=f, end="\n\n")
        
        # Record metadata
        metadata_rows.append({
            "filename"       : base_name,
            "error_type"     : error_type,
            "inject_at_move" : error_info.get("inject_at_move", ""),
            "total_moves"    : error_info.get("total_moves", ""),
            "original_move"  : error_info.get("original_move", ""),
            "corrupted_move" : error_info.get("corrupted_move", 
                               error_info.get("injected_move",
                               error_info.get("deleted_move", ""))),
            "cascade_length" : error_info.get("cascade_length", "")
        })
        
        success_count += 1
        
        # Progress update every 10,000 games
        if success_count % 10000 == 0:
            print(f"  Processed {success_count:,} corrupted games...")
    
    except Exception as e:
        error_count += 1
        continue

# ── Process clean files ──────────────────────────────────────
print(f"\nCopying {n_clean:,} clean games...")
clean_copied = 0

for filepath in clean_files:
    try:
        base_name = f"clean_{clean_copied + 1:06d}.pgn"
        shutil.copy(filepath, f"{CLEAN_DIR}{base_name}")
        
        # Also add to metadata as clean example
        metadata_rows.append({
            "filename"       : base_name,
            "error_type"     : 0,
            "inject_at_move" : "",
            "total_moves"    : "",
            "original_move"  : "",
            "corrupted_move" : "",
            "cascade_length" : ""
        })
        
        clean_copied += 1
        
    except Exception as e:
        continue

# ── Write metadata CSV ───────────────────────────────────────
print(f"\nWriting metadata CSV...")
fieldnames = [
    "filename", "error_type", "inject_at_move",
    "total_moves", "original_move", "corrupted_move", "cascade_length"
]

with open(METADATA_FILE, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(metadata_rows)

# ── Final Summary ────────────────────────────────────────────
print(f"\n{'=' * 50}")
print(f"PIPELINE COMPLETE")
print(f"{'=' * 50}")
print(f"Corrupted games saved : {success_count:,}")
print(f"Clean games saved     : {clean_copied:,}")
print(f"Skipped               : {skip_count:,}")
print(f"Errors                : {error_count:,}")
print(f"Metadata CSV          : {METADATA_FILE}")
print(f"\nOutput directories:")
print(f"  Corrupted    : {CORRUPTED_DIR}")
print(f"  Ground truth : {GROUND_TRUTH_DIR}")
print(f"  Clean        : {CLEAN_DIR}")

Total games       : 97,160
To corrupt (80%)  : 77,728
To keep clean (20%): 19,432

Error type distribution within corrupted set:
  Type 1 — Wrong move    : ~19,432
  Type 2 — OCR sub       : ~19,432
  Type 3 — Delete no shift: ~19,432
  Type 4 — Delete + shift : ~19,432

Starting pipeline...


  Processed 10,000 corrupted games...


  Processed 20,000 corrupted games...


  Processed 30,000 corrupted games...


  Processed 40,000 corrupted games...


...

  Processed 60,000 corrupted games...


  Processed 70,000 corrupted games...



Copying 19,432 clean games...



Writing metadata CSV...



PIPELINE COMPLETE
Corrupted games saved : 77,728
Clean games saved     : 19,432
Skipped               : 0
Errors                : 0
Metadata CSV          : /home/sagemaker-user/corruption_metadata.csv

Output directories:
  Corrupted    : /home/sagemaker-user/corrupted_pgn_games/
  Ground truth : /home/sagemaker-user/ground_truth_games/
  Clean        : /home/sagemaker-user/clean_pgn_games/


## 11 - Verify Metadata CSV

In [17]:
import pandas as pd

# Load and inspect metadata
df = pd.read_csv(METADATA_FILE)

print(f"Metadata shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nError type distribution:")
print(df['error_type'].value_counts().sort_index())
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nSample corrupted entry:")
print(df[df['error_type'] > 0].iloc[0])
print(f"\nSample clean entry:")
print(df[df['error_type'] == 0].iloc[0])

Metadata shape: (97160, 7)

First 5 rows:
          filename  error_type  inject_at_move  total_moves original_move  \
0  game_000001.pgn           2            38.0         71.0          Bg5+   
1  game_000002.pgn           4            22.0         29.0          Qh5+   
2  game_000003.pgn           4            11.0         38.0           Nf6   
3  game_000004.pgn           4             7.0         55.0           Bg4   
4  game_000005.pgn           2            10.0         84.0           Nc3   

  corrupted_move  cascade_length  
0           8g5+             NaN  
1           Qh5+             6.0  
2            Nf6            26.0  
3            Bg4            47.0  
4            Mc3             NaN  

Error type distribution:
error_type
0    19432
1    19432
2    19432
3    19432
4    19432
Name: count, dtype: int64

Missing values:
filename              0
error_type            0
inject_at_move    19432
total_moves       19432
original_move     19432
corrupted_move    19432
cascad

## 12 - Uploading Corrupted/Ground-truth/Clean to S3

In [19]:
import boto3
import os
import glob
from pathlib import Path

s3 = boto3.client('s3', region_name='us-east-1')
BUCKET = "chess-pgn-capstone-387100404445-us-east-1-an"

def upload_directory(local_dir, s3_prefix, file_pattern="*.pgn"):
    """Upload all files in a local directory to S3."""
    files = glob.glob(f"{local_dir}{file_pattern}")
    total = len(files)
    uploaded = 0
    
    for filepath in files:
        filename = os.path.basename(filepath)
        s3_key = f"{s3_prefix}/{filename}"
        s3.upload_file(filepath, BUCKET, s3_key)
        uploaded += 1
        if uploaded % 10000 == 0:
            print(f"  Uploaded {uploaded:,} of {total:,}...")
    
    return uploaded

print("Starting S3 upload...")
print("This will take 15-25 minutes for all files...")
print()

# Upload corrupted PGN files
print("1/4 Uploading corrupted PGN files...")
n = upload_directory(CORRUPTED_DIR, "corrupted-pgn")
print(f"    Done — {n:,} files uploaded")

# Upload ground truth files
print("2/4 Uploading ground truth files...")
n = upload_directory(GROUND_TRUTH_DIR, "ground-truth")
print(f"    Done — {n:,} files uploaded")

# Upload clean files
print("3/4 Uploading clean PGN files...")
n = upload_directory(CLEAN_DIR, "clean-pgn", file_pattern="*.pgn")
print(f"    Done — {n:,} files uploaded")

# Upload metadata CSV
print("4/4 Uploading metadata CSV...")
s3.upload_file(METADATA_FILE, BUCKET, "corruption_metadata.csv")
print(f"    Done — metadata CSV uploaded")

print()
print("=" * 50)
print("S3 UPLOAD COMPLETE")
print("=" * 50)
print(f"Bucket: {BUCKET}")
print(f"  corrupted-pgn/     : 77,728 files")
print(f"  ground-truth/      : 77,728 files")
print(f"  clean-pgn/         : 19,432 files")
print(f"  corruption_metadata.csv")

Starting S3 upload...
This will take 15-25 minutes for all files...

1/4 Uploading corrupted PGN files...


  Uploaded 10,000 of 77,728...


  Uploaded 20,000 of 77,728...


  Uploaded 30,000 of 77,728...


  Uploaded 40,000 of 77,728...


...

  Uploaded 60,000 of 77,728...


  Uploaded 70,000 of 77,728...


    Done — 77,728 files uploaded
3/4 Uploading clean PGN files...


  Uploaded 10,000 of 19,432...


    Done — 19,432 files uploaded
4/4 Uploading metadata CSV...
    Done — metadata CSV uploaded

S3 UPLOAD COMPLETE
Bucket: chess-pgn-capstone-387100404445-us-east-1-an
  corrupted-pgn/     : 77,728 files
  ground-truth/      : 77,728 files
  clean-pgn/         : 19,432 files
  corruption_metadata.csv
